# Automated Codebook

## 1. Imports

In [1]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 2. Configuration

In [2]:
MODEL_NAME = "gpt-oss:120b"
TEMPERATURE = 0.3 # taken from Nyaaba et al (2026)
NUM_CTX = 32768  # Context used for prediction. Could push higher

NUM_INTERVIEWS = 5 

# Chunking Settings
CHUNK_SIZE = 4000  # Roughly 800-1000 words per chunk to prevent "Lost in the Middle"
CHUNK_OVERLAP = 400  # Overlap across chunks

## 3. Prompt Templates

### 3.1. System prompt: A variation on Tree of Thought (ToT)

Five experts that analyse each chunk slightly differently with expert 5 coordinating.  Differs from standard ToT as no experts leave the room.

In [3]:
EXPERTS_SETUP = """Five different experts are collaboratively identifying themes across interviews regarding the reuse of a simulation model.
- Expert 1 focuses on barriers and challenges.
- Expert 2 focuses on what went well (facilitators/successes).
- Expert 3 focuses on what needs to be improved.
- Expert 4 focuses on anything unexpected or outlier observations.
- Expert 5 acts as the Cross-Checker, identifying overlaps, contradictions, and connections between the other experts' findings."""

### 3.2 Codebook prompt

The prompt will vary depending on the interview number. For interview 2 onwards the current codebook is appended. Implemented using `jinja2` if else. 

In [4]:
codebook_template = """
Method:
{{ experts_setup }}

{% if current_codebook %}
Task:
We are inductively coding interviews sequentially by going through transcripts chunk-by-chunk. Below is the **Current Codebook** generated from previous chunks.

CURRENT CODEBOOK:
{{ current_codebook }}

Process for the New Transcript Chunk:
1. Apply existing codes: If a quote from the new chunk matches an existing code, extract the exact quote and append it to that code's "Source Evidence".
2. Generate new codes: If a quote reveals a distinct, meaningful idea NOT covered by the Current Codebook, generate a new code and definition for it.
3. Refine: Expert 5 will cross-check to ensure new codes don't secretly overlap with existing ones.

Output an updated, merged Markdown table containing BOTH the old codes (with their previous quotes) and the newly added codes/quotes. 
CRITICAL: For every new quote added, you MUST append the specific identifier [{{ interview_id }}] before the quote in the Source Evidence column.

{% else %}
Process:
1. Extraction: All experts will independently read the data and write down their initial observations based on their specific roles. 
2. Discussion: They will share their findings with the group. Expert 5 will critique the findings, pointing out redundancies.
3. Refinement: Experts will debate, drop unsupported codes, and refine their remaining codes to be more precise based on feedback. 

Task:
Extract the most relevant inductively emerging codes that capture distinct and meaningful ideas about the reuse of the simulation model. 
For each emerging code, output a Markdown table with the following columns:
1. Expert Role
2. Emerging Code
3. Definition
4. Source Evidence (Extract the exact quote and MUST append the specific identifier [{{ interview_id }}] before the quote, e.g., "[{{ interview_id }}] '...exact participant quote...'")
{% endif %}

Transcript Chunk ({{ interview_id }}):
{{ transcript_chunk }}
"""

### 3.3 Setup prompt template

In [5]:
prompt = PromptTemplate.from_template(
    codebook_template, 
    template_format="jinja2"
)

## 4. Initialise Ollama and LLM

In [6]:
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    num_ctx=NUM_CTX
)

## 5. Setup chain and chunkifier

In [7]:
# Define the Chain (LCEL)

# chain = filled in prompt template -> chosen LLM -> cleaned up response
analysis_chain = prompt | llm | StrOutputParser()

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

## 6. Run analysis function

In [14]:
def run_analysis():
    current_codebook = ""
    
    for i in range(1, NUM_INTERVIEWS + 1):
        filename = f"interview_{i}.txt"
        
        if not os.path.exists(filename):
            print(f"File {filename} not found. Skipping...")
            continue
            
        with open(filename, 'r', encoding='utf-8') as f:
            interview_text = f.read()
            
        # Format a clean ID for "qualitative traceability"
        current_interview_id = filename.replace(".txt", "").replace("_", " ").title()
            
        # Split the interview into chunks to help LLM avoid "missing the middle"
        chunks = text_splitter.split_text(interview_text)
        print(f"Divided {filename} into {len(chunks)} chunks.")
        
        for chunk_idx, chunk_text in enumerate(chunks, 1):
            print(f"  -> Processing {current_interview_id} - Chunk {chunk_idx}/{len(chunks)}...")
            
            # LangChain pipeline with the mapped variables
            current_codebook = analysis_chain.invoke({
                "experts_setup": EXPERTS_SETUP,
                "current_codebook": current_codebook,
                "transcript_chunk": chunk_text,
                "interview_id": current_interview_id
            })
            
            # Save progress after EVERY chunk to prevent data loss on long runs
            output_filename = f"../output/codebook_after_interview_{i}_chunk_{chunk_idx}.md"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(current_codebook)
                
        print(f"✅ Finished all chunks for {filename}.")
        
    print(f"\n ✅ Codebook complete")

In [ ]:
#run_analysis()